

---

# **TRABAJO PARCIAL - TEORÍA DE COMPILADORES**
Sección : 13962



---


**INTEGRANTES**

Vanessa Barrientos Villalta - U202413287

Matias Javier Del Castillo Mendoza - U202411305

Vivianne Fatima Ríos Hasegawa - U202410135



---



**Instalación de ANTLR**

In [ ]:
!pip install antlr4-python3-runtime==4.13.2 -q
!curl -s -O https://www.antlr.org/download/antlr-4.13.2-complete.jar

**ANALIZADOR LÉXICO**

Creación del archivo "SoundSpecLexer.g4"

In [ ]:
grammar = r'''
lexer grammar SoundSpecLexer;

// --- KEYWORDS ---
TRACK     : 'track';
EFFECT    : 'effect';
MIX       : 'mix';
BPM       : 'bpm';
WAVE      : 'wave';
DELAY     : 'delay';
DECAY     : 'decay';
NOTES     : 'notes';

// --- VALORES TIPADOS Y ENUMS ---
WAVE_TYPE    : 'SAWTOOTH' | 'SINE' | 'SQUARE' | 'TRIANGLE';
NOTE         : [A-G][#b]?[0-8];
FRACTION     : [0-9]+ '/' [0-9]+;
TIME_MS      : [0-9]+ 'ms';

// --- OPERADORES Y SÍMBOLOS ---
PIPE_OP      : '->';
ASSIGN       : '=';
COLON        : ':';
SEMI         : ';';
COMMA        : ',';
LBRACE       : '{';
RBRACE       : '}';
LBRACK       : '[';
RBRACK       : ']';

// --- LITERALES NUMÉRICOS E IDENTIFICADORES ---
FLOAT : [0-9]+ '.' [0-9]+;
INT   : [0-9]+;
ID    : [a-zA-Z_][a-zA-Z0-9_]*;

// --- REGLAS AUXILIARES ---
WS           : [ \t\r\n]+ -> skip;
COMMENT      : '//' ~[\r\n]* -> skip;
'''

with open("SoundSpecLexer.g4", "w") as f:
    f.write(grammar)

Compilar con ANTLR4

In [ ]:
!java -jar antlr-4.13.2-complete.jar -Dlanguage=Python3 SoundSpecLexer.g4

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


Pruebas del analizador léxico

In [ ]:
from antlr4 import InputStream
from antlr4.error.ErrorListener import ErrorListener
from SoundSpecLexer import SoundSpecLexer

class ListenerErrores(ErrorListener):
    def __init__(self, fase):
        super().__init__()
        self.fase = fase
        self.errores = []

    def syntaxError(self, recognizer, offendingSymbol, line, column, msg, e):
        self.errores.append({
            "fase": self.fase,
            "linea": line,
            "columna": column + 1,
            "mensaje": msg,
        })

def probar_analizador_lexico(cod_fuente, Nprueba):
    print(Nprueba)
    print("\nCódigo de entrada:" + cod_fuente)

    input_stream = InputStream(cod_fuente)
    lexer = SoundSpecLexer(input_stream)
    errores = ListenerErrores("LÉXICO")
    lexer.removeErrorListeners()
    lexer.addErrorListener(errores)
    tokens = lexer.getAllTokens()

    print(f"{'TOKEN':<18} | {'LEXEMA'}")
    print("-" * 40)
    for token in tokens:
        token_name = lexer.symbolicNames[token.type]
        print(f"{token_name:<18} | '{token.text}'")
    for error in errores.errores:
        print(f"[{error['fase']}] línea {error['linea']}, columna {error['columna']}: {error['mensaje']}")
    print("\n" + "=" * 50 + "\n")

# --- PRUEBA 1: Declaración de Track Básica ---
prueba1 = """
track BassLine {
    bpm: 120;
    wave: SAWTOOTH;
    notes: [ C2: 1/4, E2: 1/4, G2: 1/2 ];
}
"""

# --- PRUEBA 2: Declaración de Effect y Mix Simple ---
prueba2 = """
effect Reverb1 {
    delay: 200ms;
    decay: 0.5;
}

mix Song = BassLine -> Reverb1;
"""

# --- PRUEBA 3: Cadena de Efectos en Serie (Procesamiento Múltiple) ---
prueba3 = """
track SubBass {
    bpm: 120;
    wave: SINE;
    notes: [ C1: 1/2, G1: 1/2 ];
}

effect DelayCorto {
    delay: 150ms;
    decay: 0.4;
}

effect EcoLargo {
    delay: 400ms;
    decay: 0.2;
}

mix BajoProcesado = SubBass -> DelayCorto -> EcoLargo;
"""

# --- PRUEBA 4: Polifonía/Multi-Track y Diferentes Ondas ---
prueba4 = """
track LeadSintetizador {
    bpm: 140;
    wave: SQUARE;
    notes: [ C4: 1/8, E4: 1/8, G4: 1/4, B4: 1/2 ];
}

track PadAtmósfera {
    bpm: 140;
    wave: TRIANGLE;
    notes: [ C3: 1/1 ];
}

effect EspacioGrande {
    delay: 500ms;
    decay: 0.6;
}

mix MelodiaPrincipal = LeadSintetizador -> EspacioGrande;
mix FondoAmbiente = PadAtmósfera;
"""

# Ejecutar las pruebas
probar_analizador_lexico(prueba1, "PRUEBA 1: Definición de Pista Musical Básica")
probar_analizador_lexico(prueba2, "PRUEBA 2: Definición de Efecto y Mezcla Simple")
probar_analizador_lexico(prueba3, "PRUEBA 3: Cadena de Efectos Múltiples (Serie)")
probar_analizador_lexico(prueba4, "PRUEBA 4: Configuración Multi-Pista y Tipos de Onda")

PRUEBA 1: Definición de Pista Musical Básica

Código de entrada:
track BassLine {
    bpm: 120;
    wave: SAWTOOTH;
    notes: [ C2: 1/4, E2: 1/4, G2: 1/2 ];
}

TOKEN              | LEXEMA
----------------------------------------
TRACK              | 'track'
ID                 | 'BassLine'
LBRACE             | '{'
BPM                | 'bpm'
COLON              | ':'
INT                | '120'
SEMI               | ';'
WAVE               | 'wave'
COLON              | ':'
WAVE_TYPE          | 'SAWTOOTH'
SEMI               | ';'
NOTES              | 'notes'
COLON              | ':'
LBRACK             | '['
NOTE               | 'C2'
COLON              | ':'
FRACTION           | '1/4'
COMMA              | ','
NOTE               | 'E2'
COLON              | ':'
FRACTION           | '1/4'
COMMA              | ','
NOTE               | 'G2'
COLON              | ':'
FRACTION           | '1/2'
RBRACK             | ']'
SEMI               | ';'
RBRACE             | '}'


PRUEBA 2: Definición de Efecto

**Observaciones del analizador léxico**

- `PadAtmósfera` produce un error léxico porque `ID` solo admite letras ASCII, dígitos y guion bajo (`[a-zA-Z_][a-zA-Z0-9_]*`). La `ó` no pertenece a ningún token.
- Los nombres con forma de nota (`A1`, `C4`, `Bb2`) se reconocen como `NOTE` y los tipos de onda (`SINE`, `SQUARE`, `TRIANGLE`, `SAWTOOTH`) como `WAVE_TYPE`, porque esas reglas se declaran antes que `ID`. Por eso funcionan como palabras reservadas y no pueden usarse como identificadores.

**ANALIZADOR SINTÁCTICO**

El analizador verifica la estructura de SoundSpec con los tokens del analizador léxico y las mismas reglas utilizadas por el analizador semántico. Si el programa es válido, devuelve su árbol sintáctico; si hay errores, indica la fase, la línea y la columna (numeradas desde 1).

| Elemento | Estructura admitida |
| --- | --- |
| Programa | Una o más declaraciones `track`, `effect` o `mix`, seguidas del fin de la entrada. |
| Pista | `track ID { ... }`, con una o más propiedades `bpm`, `wave` o `notes`. |
| Propiedades de pista | `bpm: INT;`, `wave: WAVE_TYPE;` y `notes: [ ... ];`. |
| Notas | Una o más notas separadas por comas; cada nota tiene una fracción de duración y puede llevar `:`: `C4:1/4` o `C4 1/4`. |
| Efecto | `effect ID { ... }`, con una o más propiedades `delay: TIME_MS;` o `decay: FLOAT;`. |
| Mezcla | `mix ID = ID (-> ID)*;`. |

**Orden de ejecución:** ejecutar las celdas en orden desde la instalación de ANTLR y la creación del lexer. La gramática `SoundSpecParser.g4` se genera una sola vez y su parser se comparte con el analizador semántico.

La existencia de los identificadores, el rango audible y la suma de duraciones corresponden al análisis semántico. Las propiedades pueden aparecer en cualquier orden. El programa debe contener al menos una declaración; los bloques de pistas y efectos deben contener al menos una propiedad.


Creación del archivo `SoundSpecParser.g4` (lo usan el analizador sintáctico y el semántico)

In [ ]:
parser_grammar = r"""
parser grammar SoundSpecParser;
options { tokenVocab=SoundSpecLexer; }

program    : statement+ EOF ;
statement  : trackDecl | effectDecl | mixRule ;

trackDecl  : TRACK ID LBRACE trackProp+ RBRACE ;
trackProp  : BPM COLON INT SEMI
           | WAVE COLON WAVE_TYPE SEMI
           | NOTES COLON noteList SEMI ;
noteList   : LBRACK noteItem (COMMA noteItem)* RBRACK ;
noteItem   : NOTE COLON? FRACTION ;

effectDecl : EFFECT ID LBRACE effectProp+ RBRACE ;
effectProp : DELAY COLON TIME_MS SEMI
           | DECAY COLON FLOAT SEMI ;

mixRule    : MIX ID ASSIGN ID (PIPE_OP ID)* SEMI ;
"""

with open("SoundSpecParser.g4", "w", encoding="utf-8") as f:
    f.write(parser_grammar)
print("Gramática compartida SoundSpecParser.g4 creada.")


Gramática compartida SoundSpecParser.g4 creada.


Compilar con ANTLR4 (también genera `SoundSpecParserListener`, que usa el analizador semántico)

In [ ]:
!java -jar antlr-4.13.2-complete.jar -Dlanguage=Python3 -listener SoundSpecParser.g4

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


Implementación del analizador y reporte de errores

In [ ]:
from antlr4 import InputStream, CommonTokenStream
from SoundSpecLexer import SoundSpecLexer
from SoundSpecParser import SoundSpecParser

def analizar_sintaxis(cod_fuente):
    """Valida toda la entrada. No realiza comprobaciones semánticas."""
    lexer = SoundSpecLexer(InputStream(cod_fuente))
    errores_lexicos = ListenerErrores("LÉXICO")
    lexer.removeErrorListeners()
    lexer.addErrorListener(errores_lexicos)
    tokens = CommonTokenStream(lexer)
    tokens.fill()

    # Un carácter rechazado por el lexer no debe desaparecer silenciosamente.
    if errores_lexicos.errores:
        return {"valido": False, "errores": errores_lexicos.errores,
                "arbol": None, "arbol_texto": None}

    tokens.seek(0)
    parser = SoundSpecParser(tokens)
    errores_sintacticos = ListenerErrores("SINTÁCTICO")
    parser.removeErrorListeners()
    parser.addErrorListener(errores_sintacticos)
    arbol = parser.program()
    valido = parser.getNumberOfSyntaxErrors() == 0
    return {
        "valido": valido,
        "errores": errores_sintacticos.errores,
        "arbol": arbol if valido else None,
        "arbol_texto": arbol.toStringTree(recog=parser) if valido else None,
    }

def probar_analizador_sintactico(cod_fuente, titulo, mostrar_arbol=True):
    print(titulo)
    print("Código de entrada:\n" + cod_fuente)
    resultado = analizar_sintaxis(cod_fuente)
    if resultado["valido"]:
        print("Sin errores sintácticos.")
        if mostrar_arbol:
            print("Árbol sintáctico:", resultado["arbol_texto"])
    else:
        for error in resultado["errores"]:
            print(f"[{error['fase']}] línea {error['linea']}, "
                  f"columna {error['columna']}: {error['mensaje']}")
        print("La entrada no se acepta; no se devuelve un árbol válido.")
    print("\n" + "." * 50 + "\n")
    return resultado

Pruebas del analizador sintáctico: entradas válidas

In [ ]:
casos_sintacticos_validos = [
    ("Pista con notas y dos puntos", """
track BassLine {
    bpm: 120;
    wave: SAWTOOTH;
    notes: [C2:1/4, E2:1/4, G2:1/2];
}
"""),
    ("Notas sin dos puntos, admitidas por la gramática", """
track Melodia {
    bpm: 140;
    wave: SINE;
    notes: [C4 1/2, G4 1/2];
}
"""),
    ("Programa completo con cadena de efectos", """
track SubBass { bpm: 120; wave: SINE; notes: [C1:1/2, G1:1/2]; }
effect DelayCorto { delay: 150ms; decay: 0.3; }
effect EcoLargo { delay: 450ms; decay: 0.7; }
mix BajoProcesado = SubBass -> DelayCorto -> EcoLargo;
"""),
    ("Comentarios, sostenidos y bemoles", """
// El lexer reconoce alteraciones y omite comentarios.
track Alteraciones { notes: [C#4:1/2, Db4:1/2]; wave: TRIANGLE; bpm: 100; }
mix Salida = Alteraciones;
"""),
    ("Propiedades en otro orden", "track Orden { wave: SQUARE; bpm: 90; }"),
]

for titulo, codigo in casos_sintacticos_validos:
    resultado = probar_analizador_sintactico(codigo, titulo)
    assert resultado["valido"] and not resultado["errores"], titulo

print(f"Pruebas válidas superadas: {len(casos_sintacticos_validos)}.")

Pista con notas y dos puntos
Código de entrada:

track BassLine {
    bpm: 120;
    wave: SAWTOOTH;
    notes: [C2:1/4, E2:1/4, G2:1/2];
}

Sin errores sintácticos.
Árbol sintáctico: (program (statement (trackDecl track BassLine { (trackProp bpm : 120 ;) (trackProp wave : SAWTOOTH ;) (trackProp notes : (noteList [ (noteItem C2 : 1/4) , (noteItem E2 : 1/4) , (noteItem G2 : 1/2) ]) ;) })) <EOF>)

..................................................

Notas sin dos puntos, admitidas por la gramática
Código de entrada:

track Melodia {
    bpm: 140;
    wave: SINE;
    notes: [C4 1/2, G4 1/2];
}

Sin errores sintácticos.
Árbol sintáctico: (program (statement (trackDecl track Melodia { (trackProp bpm : 140 ;) (trackProp wave : SINE ;) (trackProp notes : (noteList [ (noteItem C4 1/2) , (noteItem G4 1/2) ]) ;) })) <EOF>)

..................................................

Programa completo con cadena de efectos
Código de entrada:

track SubBass { bpm: 120; wave: SINE; notes: [C1:1/2, G1:1/2]; }

Pruebas del analizador sintáctico: entradas inválidas

Se comprueban delimitadores, tipos de tokens, listas de notas, cadenas de efectos y consumo completo de la entrada. También se incluyen dos errores léxicos para asegurar que no se acepten caracteres desconocidos. Con el lexer existente, los identificadores solo admiten letras ASCII, dígitos y guion bajo; por eso `PadAtmósfera` produce un error léxico, no un error sintáctico.

In [ ]:
casos_sintacticos_invalidos = [
    ("Programa vacío", "", "SINTÁCTICO"),
    ("Pista sin propiedades", "track Melodia { }", "SINTÁCTICO"),
    ("Asignación incorrecta de una propiedad", "track Melodia { wave = SINE; }", "SINTÁCTICO"),
    ("Falta punto y coma", "track Melodia { bpm: 120 }", "SINTÁCTICO"),
    ("Falta llave de cierre", "track Melodia { bpm: 120;", "SINTÁCTICO"),
    ("Falta coma entre notas", "track Melodia { notes: [C4:1/2 G4:1/2]; }", "SINTÁCTICO"),
    ("Lista de notas vacía", "track Melodia { notes: []; }", "SINTÁCTICO"),
    ("Bloque sin propiedades", "effect Eco { }", "SINTÁCTICO"),
    ("Falta identificador después de ->", "mix Salida = Melodia -> ;", "SINTÁCTICO"),
    ("BPM con token FLOAT en lugar de INT", "track Melodia { bpm: 120.5; }", "SINTÁCTICO"),
    ("Separador de nota duplicado", "track Melodia { notes: [C4::1/1]; }", "SINTÁCTICO"),
    ("Texto extra al final del programa", "track Melodia { bpm: 120; } sobrante", "SINTÁCTICO"),
    ("Carácter no reconocido", "track Melodia { bpm: 120; } @", "LÉXICO"),
    ("Identificador con tilde no admitida por el lexer", "track PadAtmósfera { bpm: 120; }", "LÉXICO"),
]

for titulo, codigo, fase in casos_sintacticos_invalidos:
    resultado = probar_analizador_sintactico(codigo, titulo)
    assert not resultado["valido"], titulo
    assert resultado["arbol"] is None, titulo
    assert resultado["errores"], titulo
    assert all(e["fase"] == fase for e in resultado["errores"]), titulo

print(f"Pruebas de rechazo superadas: {len(casos_sintacticos_invalidos)}.")

Programa vacío
Código de entrada:

[SINTÁCTICO] línea 1, columna 1: mismatched input '<EOF>' expecting {'track', 'effect', 'mix'}
La entrada no se acepta; no se devuelve un árbol válido.

..................................................

Pista sin propiedades
Código de entrada:
track Melodia { }
[SINTÁCTICO] línea 1, columna 17: mismatched input '}' expecting {'bpm', 'wave', 'notes'}
La entrada no se acepta; no se devuelve un árbol válido.

..................................................

Asignación incorrecta de una propiedad
Código de entrada:
track Melodia { wave = SINE; }
[SINTÁCTICO] línea 1, columna 22: mismatched input '=' expecting ':'
La entrada no se acepta; no se devuelve un árbol válido.

..................................................

Falta punto y coma
Código de entrada:
track Melodia { bpm: 120 }
[SINTÁCTICO] línea 1, columna 26: missing ';' at '}'
La entrada no se acepta; no se devuelve un árbol válido.

..................................................

Falta

Pruebas de separación entre sintaxis y semántica

Estas entradas tienen estructura válida, aunque el analizador semántico existente debe reportar una referencia no declarada (SEM-02), una nota fuera del rango audible (SEM-03) o un compás incompleto (SEM-04). No se rechazan en la fase sintáctica.

In [ ]:
casos_sintacticos_semantica = [
    ("SEM-02: referencia no declarada", "mix Salida = Desconocida;"),
    ("SEM-03: nota fuera del rango audible", "track Grave { notes: [C0:1/1]; }"),
    ("SEM-04: duraciones que no completan el compás", "track Incompleta { notes: [C4:1/4, E4:1/2]; }"),
]

for titulo, codigo in casos_sintacticos_semantica:
    resultado = probar_analizador_sintactico(
        codigo, titulo, mostrar_arbol=False
    )
    assert resultado["valido"], titulo
    print("La comprobación indicada corresponde al analizador semántico.\n")

total_pruebas = (
    len(casos_sintacticos_validos) + len(casos_sintacticos_invalidos)
    + len(casos_sintacticos_semantica)
)
print(f"Total: {total_pruebas} pruebas del analizador sintáctico superadas.")

SEM-02: referencia no declarada
Código de entrada:
mix Salida = Desconocida;
Sin errores sintácticos.

..................................................

La comprobación indicada corresponde al analizador semántico.

SEM-03: nota fuera del rango audible
Código de entrada:
track Grave { notes: [C0:1/1]; }
Sin errores sintácticos.

..................................................

La comprobación indicada corresponde al analizador semántico.

SEM-04: duraciones que no completan el compás
Código de entrada:
track Incompleta { notes: [C4:1/4, E4:1/2]; }
Sin errores sintácticos.

..................................................

La comprobación indicada corresponde al analizador semántico.

Total: 22 pruebas del analizador sintáctico superadas.


**ANALIZADOR SEMÁNTICO**

Errores semánticos verificados:

| Código | Error | Regla |
|---|---|---|
| SEM-01 | Identificador redeclarado | Cada `track`, `effect` y `mix` tiene un nombre único |
| SEM-02 | Referencia no declarada | La pista y los efectos de un `mix` deben estar declarados |
| SEM-03 | Nota fuera del rango audible | 20 Hz ≤ frecuencia ≤ 20 000 Hz |
| SEM-04 | Compás incompleto | Las duraciones de `notes` deben completar compases enteros |
| SEM-05 | Categoría incorrecta en `mix` | La fuente debe ser un `track` y la cadena solo admite `effect` |
| SEM-06 | Duración inválida | El numerador y el denominador de la duración deben ser mayores que 0 |

Usa el mismo `SoundSpecParser.g4` compilado en la sección sintáctica.

Analizador semántico (tabla de símbolos y dos recorridos del árbol: el primero registra las declaraciones y el segundo valida notas y mezclas)

In [ ]:
from fractions import Fraction
from antlr4 import ParseTreeWalker
from SoundSpecParserListener import SoundSpecParserListener

SEMITONO = {'C': 0, 'D': 2, 'E': 4, 'F': 5, 'G': 7, 'A': 9, 'B': 11}
ALTERACION = {'': 0, '#': 1, 'b': -1}

def frecuencia(nota): # C#4
    # f = 440 * 2^((midi - 69) / 12), con A4 = 440 Hz (midi 69)
    midi = 12 * (int(nota[-1]) + 1) + SEMITONO[nota[0]] + ALTERACION[nota[1:-1]]
    return 440 * 2 ** ((midi - 69) / 12)

class Declaraciones(SoundSpecParserListener):
    # Pasada 1: llena la tabla de símbolos {nombre: (categoría, línea)} y detecta SEM-01
    def __init__(self, tabla, errores):
        self.tabla, self.errores = tabla, errores

    def declarar(self, token, categoria):
        if token.text in self.tabla:
            cat, linea = self.tabla[token.text]
            self.errores.append(f"[SEM-01] línea {token.line}: '{token.text}' ya fue declarado como {cat} en la línea {linea}.")
        else:
            self.tabla[token.text] = (categoria, token.line)

    def enterTrackDecl(self, ctx):  self.declarar(ctx.ID().symbol, 'track')
    def enterEffectDecl(self, ctx): self.declarar(ctx.ID().symbol, 'effect')
    def enterMixRule(self, ctx):    self.declarar(ctx.ID(0).symbol, 'mix')

class Verificador(SoundSpecParserListener):
    # Pasada 2: detecta SEM-02 a SEM-06
    def __init__(self, tabla, errores):
        self.tabla, self.errores = tabla, errores

    def enterNoteList(self, ctx):
        total = Fraction(0)
        for item in ctx.noteItem():
            nota = item.NOTE().getText()
            linea = item.start.line
            f = frecuencia(nota)
            if not 20 <= f <= 20000:
                self.errores.append(f"[SEM-03] línea {linea}: la nota {nota} ({f:.2f} Hz) está fuera del rango audible.")
            duracion = item.FRACTION().getText()
            num, den = map(int, duracion.split('/'))
            if num == 0 or den == 0:
                # La nota no se suma al compás porque su duración no es válida
                self.errores.append(f"[SEM-06] línea {linea}: la nota {nota} tiene una duración inválida ({duracion}).")
            else:
                total += Fraction(num, den)
        if total.denominator != 1:
            self.errores.append(f"[SEM-04] línea {ctx.start.line}: las duraciones suman {total} de compás; falta {1 - total % 1}.")

    def enterMixRule(self, ctx):
        ids, linea = ctx.ID(), ctx.start.line
        fuente = ids[1].getText()
        if fuente not in self.tabla:
            self.errores.append(f"[SEM-02] línea {linea}: la pista '{fuente}' no ha sido declarada.")
        elif self.tabla[fuente][0] != 'track':
            self.errores.append(f"[SEM-05] línea {linea}: '{fuente}' fue declarado como {self.tabla[fuente][0]}, pero la fuente de un mix debe ser un track.")
        for efecto in ids[2:]:
            nombre = efecto.getText()
            if nombre not in self.tabla:
                self.errores.append(f"[SEM-02] línea {linea}: el efecto '{nombre}' no ha sido declarado.")
            elif self.tabla[nombre][0] != 'effect':
                self.errores.append(f"[SEM-05] línea {linea}: '{nombre}' fue declarado como {self.tabla[nombre][0]}, pero la cadena de un mix solo admite effects.")

def analizador_semantico(cod_fuente, Nprueba):
    print(Nprueba)
    print("\nCódigo de entrada:" + cod_fuente)
    resultado = analizar_sintaxis(cod_fuente)
    if not resultado["valido"]:
        errores = [f"[{e['fase']}] línea {e['linea']}, columna {e['columna']}: {e['mensaje']}"
                   for e in resultado["errores"]]
        errores.append("No se ejecuta el análisis semántico.")
    else:
        tabla, errores = {}, []
        ParseTreeWalker().walk(Declaraciones(tabla, errores), resultado["arbol"])
        ParseTreeWalker().walk(Verificador(tabla, errores), resultado["arbol"])
        errores.sort(key=lambda e: int(e.split("línea ")[1].split(":")[0]))

    print("\n".join(errores) if errores else "Sin errores semánticos.")
    print("\n" + "." * 50 + "\n")

Pruebas del analizador semántico

In [ ]:
prueba1 = """
mix Song = BassLine -> Reverb1;
track BassLine {
    bpm: 120;
    wave: SAWTOOTH;
    notes: [ C#4 1/2, Db4 1/4, Db4 1/4 ];
}
effect Reverb1 { delay: 200ms; decay: 0.5; }
"""

prueba2 = """
track Melodia { notes: [ C4 1/1 ]; }
effect Melodia { decay: 0.5; }
"""

prueba3 = """
track Melodia { notes: [ C4 1/1 ]; }
effect Eco { delay: 300ms; }
mix Salida = Guitarra -> Chorus;
mix Final = Eco;
mix Doble = Melodia -> Melodia;
"""

prueba4 = """
track Alteraciones { notes: [ C#4 1/2, Db4 1/4, D#0 1/4 ]; }
"""

prueba5 = """
track Melodia { notes: [ C4 1/4, D4 1/4, E4 1/4 ]; }
track Ritmo { notes: [ C4 1/0, D4 0/4, E4 1/1 ]; }
"""

prueba6 = """
track Melodia { wave = SINE; }
"""

prueba7 = """
track Melodia { bpm: 120; } @
"""

analizador_semantico(prueba1, "PRUEBA 1: Programa correcto (mix antes de sus declaraciones, con sostenidos y bemoles)")
analizador_semantico(prueba2, "PRUEBA 2: Identificador redeclarado (SEM-01)")
analizador_semantico(prueba3, "PRUEBA 3: Referencias no declaradas (SEM-02) y de categoría incorrecta (SEM-05)")
analizador_semantico(prueba4, "PRUEBA 4: Nota fuera del rango audible con alteraciones (SEM-03)")
analizador_semantico(prueba5, "PRUEBA 5: Compás incompleto (SEM-04) y duraciones inválidas (SEM-06)")
analizador_semantico(prueba6, "PRUEBA 6: Error sintáctico")
analizador_semantico(prueba7, "PRUEBA 7: Error léxico (carácter no reconocido)")

PRUEBA 1: Programa correcto (mix antes de sus declaraciones, con sostenidos y bemoles)

Código de entrada:
mix Song = BassLine -> Reverb1;
track BassLine {
    bpm: 120;
    wave: SAWTOOTH;
    notes: [ C#4 1/2, Db4 1/4, Db4 1/4 ];
}
effect Reverb1 { delay: 200ms; decay: 0.5; }

Sin errores semánticos.

..................................................

PRUEBA 2: Identificador redeclarado (SEM-01)

Código de entrada:
track Melodia { notes: [ C4 1/1 ]; }
effect Melodia { decay: 0.5; }

[SEM-01] línea 3: 'Melodia' ya fue declarado como track en la línea 2.

..................................................

PRUEBA 3: Referencias no declaradas (SEM-02) y de categoría incorrecta (SEM-05)

Código de entrada:
track Melodia { notes: [ C4 1/1 ]; }
effect Eco { delay: 300ms; }
mix Salida = Guitarra -> Chorus;
mix Final = Eco;
mix Doble = Melodia -> Melodia;

[SEM-02] línea 4: la pista 'Guitarra' no ha sido declarada.
[SEM-02] línea 4: el efecto 'Chorus' no ha sido declarado.
[SEM-05] línea 5:

## DEMOSTRACIÓN FINAL: Estrellita, ¿dónde estás?

Programa completo en **SoundSpec** de la melodía *Twinkle, Twinkle, Little Star*: 42 notas, 12 compases, tempo de 100 BPM y onda SINE. Incluye una versión sin efectos y otra con eco.

Ejecuta las dos celdas siguientes después de las secciones de ANTLR y de los analizadores. Se reutilizan el lexer, el parser y los listeners semánticos del proyecto. Esta demostración valida el programa y muestra su árbol de análisis; no genera audio.


In [ ]:
# Script SoundSpec de la canción completa.
cancion_estrellita = r"""
// CANCION: Estrellita, donde estas?
// MELODIA: Twinkle, Twinkle, Little Star
// Tonalidad: Do mayor

track MelodiaEstrellita {
    bpm: 100;
    wave: SINE;
    notes: [
        // Frase 1: Do Do Sol Sol La La Sol
        C4:1/4,
        C4:1/4,
        G4:1/4,
        G4:1/4,
        A4:1/4,
        A4:1/4,
        G4:1/2,

        // Frase 2: Fa Fa Mi Mi Re Re Do
        F4:1/4,
        F4:1/4,
        E4:1/4,
        E4:1/4,
        D4:1/4,
        D4:1/4,
        C4:1/2,

        // Frase 3: Sol Sol Fa Fa Mi Mi Re
        G4:1/4,
        G4:1/4,
        F4:1/4,
        F4:1/4,
        E4:1/4,
        E4:1/4,
        D4:1/2,

        // Frase 4: Repeticion de la frase anterior
        G4:1/4,
        G4:1/4,
        F4:1/4,
        F4:1/4,
        E4:1/4,
        E4:1/4,
        D4:1/2,

        // Frase 5: Regreso a la melodia inicial
        C4:1/4,
        C4:1/4,
        G4:1/4,
        G4:1/4,
        A4:1/4,
        A4:1/4,
        G4:1/2,

        // Frase 6: Cierre de la cancion
        F4:1/4,
        F4:1/4,
        E4:1/4,
        E4:1/4,
        D4:1/4,
        D4:1/4,
        C4:1/2
    ];
}

// Efecto de eco suave.
effect EcoSuave {
    delay: 180ms;
    decay: 0.25;
}

// Version sin efectos.
mix EstrellitaOriginal = MelodiaEstrellita;

// Version con eco.
mix EstrellitaConEco = MelodiaEstrellita -> EcoSuave;
""".strip()

print(cancion_estrellita)


// CANCION: Estrellita, donde estas?
// MELODIA: Twinkle, Twinkle, Little Star
// Tonalidad: Do mayor

track MelodiaEstrellita {
    bpm: 100;
    wave: SINE;
    notes: [
        // Frase 1: Do Do Sol Sol La La Sol
        C4:1/4,
        C4:1/4,
        G4:1/4,
        G4:1/4,
        A4:1/4,
        A4:1/4,
        G4:1/2,

        // Frase 2: Fa Fa Mi Mi Re Re Do
        F4:1/4,
        F4:1/4,
        E4:1/4,
        E4:1/4,
        D4:1/4,
        D4:1/4,
        C4:1/2,

        // Frase 3: Sol Sol Fa Fa Mi Mi Re
        G4:1/4,
        G4:1/4,
        F4:1/4,
        F4:1/4,
        E4:1/4,
        E4:1/4,
        D4:1/2,

        // Frase 4: Repeticion de la frase anterior
        G4:1/4,
        G4:1/4,
        F4:1/4,
        F4:1/4,
        E4:1/4,
        E4:1/4,
        D4:1/2,

        // Frase 5: Regreso a la melodia inicial
        C4:1/4,
        C4:1/4,
        G4:1/4,
        G4:1/4,
        A4:1/4,
        A4:1/4,
        G4:1/2,

        // Frase 6: Cierre de la c

In [ ]:
# Validación con los analizadores existentes del proyecto.
from fractions import Fraction
from pathlib import Path
from html import escape
from antlr4 import InputStream, CommonTokenStream, ParseTreeWalker, Token
from IPython.display import HTML, display

requisitos_estrellita = (
    "cancion_estrellita", "analizar_sintaxis", "SoundSpecLexer",
    "Declaraciones", "Verificador"
)
faltantes_estrellita = [nombre for nombre in requisitos_estrellita if nombre not in globals()]
if faltantes_estrellita:
    raise RuntimeError("Ejecuta primero las celdas anteriores del notebook. Faltan: "
                       + ", ".join(faltantes_estrellita))

resultado_estrellita = analizar_sintaxis(cancion_estrellita)
print("ESTRELLITA, ¿DÓNDE ESTÁS? — VALIDACIÓN DE SOUNDSPEC")
print("=" * 60)

if not resultado_estrellita["valido"]:
    for error in resultado_estrellita["errores"]:
        print(f"[{error['fase']}] línea {error['linea']}, "
              f"columna {error['columna']}: {error['mensaje']}")
    print("Programa rechazado. No se ejecuta el análisis semántico.")
else:
    print("[LÉXICO] Sin errores: todos los caracteres forman tokens válidos.")
    print("[SINTÁCTICO] Sin errores: la entrada completa cumple la gramática.")

    tabla_estrellita, errores_estrellita = {}, []
    ParseTreeWalker().walk(Declaraciones(tabla_estrellita, errores_estrellita),
                          resultado_estrellita["arbol"])
    ParseTreeWalker().walk(Verificador(tabla_estrellita, errores_estrellita),
                          resultado_estrellita["arbol"])

    if errores_estrellita:
        for error in errores_estrellita:
            print(error)
        print("Programa rechazado por errores semánticos.")
    else:
        print("[SEMÁNTICO] Sin errores: declaraciones, referencias, notas y duraciones válidas.")
        pista_estrellita = resultado_estrellita["arbol"].statement(0).trackDecl()
        lista_estrellita = next(prop.noteList() for prop in pista_estrellita.trackProp()
                               if prop.noteList() is not None)
        notas_estrellita = lista_estrellita.noteItem()
        compases_estrellita = sum(
            (Fraction(nota.FRACTION().getText()) for nota in notas_estrellita), Fraction(0)
        )
        tokens_estrellita = CommonTokenStream(SoundSpecLexer(InputStream(cancion_estrellita)))
        tokens_estrellita.fill()
        cantidad_tokens_estrellita = sum(token.type != Token.EOF for token in tokens_estrellita.tokens)

        print(f"\nResumen: {len(notas_estrellita)} notas | {compases_estrellita} compases "
              f"| {cantidad_tokens_estrellita} tokens (sin EOF)")
        print("Tempo: 100 BPM | Onda: SINE | Efecto: delay 180ms, decay 0.25")
        print("\nTabla de símbolos:")
        for nombre, (categoria, linea) in tabla_estrellita.items():
            print(f"  {nombre:<22} {categoria:<8} línea {linea}")

        Path("estrellita.soundspec").write_text(cancion_estrellita + "\n", encoding="utf-8")
        Path("estrellita_arbol.txt").write_text(resultado_estrellita["arbol_texto"], encoding="utf-8")
        print("\nPROGRAMA ACEPTADO POR LAS TRES FASES.")
        print("Archivos en el entorno de Colab: estrellita.soundspec y estrellita_arbol.txt")
        display(HTML("<details><summary>Ver árbol de análisis sintáctico de ANTLR</summary>"
                     "<pre style='white-space:pre-wrap'>"
                     + escape(resultado_estrellita["arbol_texto"]) + "</pre></details>"))


ESTRELLITA, ¿DÓNDE ESTÁS? — VALIDACIÓN DE SOUNDSPEC
[LÉXICO] Sin errores: todos los caracteres forman tokens válidos.
[SINTÁCTICO] Sin errores: la entrada completa cumple la gramática.
[SEMÁNTICO] Sin errores: declaraciones, referencias, notas y duraciones válidas.

Resumen: 42 notas | 12 compases | 208 tokens (sin EOF)
Tempo: 100 BPM | Onda: SINE | Efecto: delay 180ms, decay 0.25

Tabla de símbolos:
  MelodiaEstrellita      track    línea 5
  EcoSuave               effect   línea 66
  EstrellitaOriginal     mix      línea 72
  EstrellitaConEco       mix      línea 75

PROGRAMA ACEPTADO POR LAS TRES FASES.
Archivos en el entorno de Colab: estrellita.soundspec y estrellita_arbol.txt


**PLAN DE VALIDACIÓN DEL COMPILADOR**

El plan comprueba que cada fase del compilador acepte los programas correctos y rechace los incorrectos en la fase que corresponde, con el código de error y la línea esperados. Cada caso define su entrada y su resultado esperado; las celdas siguientes ejecutan todos los casos con el mismo lexer, parser y listeners semánticos del proyecto y comparan lo obtenido con lo esperado.

| Categoría | Objetivo | Casos | Resultado esperado |
|---|---|---|---|
| Aceptación (A) | Programas válidos que usan todas las construcciones, incluidos valores frontera | 9 | Pasan las tres fases sin errores |
| Léxico (L) | Caracteres que no pertenecen a ningún token | 3 | Rechazo en la fase LÉXICO |
| Sintáctico (S) | Delimitadores, tipos de token, listas y consumo hasta EOF | 11 | Rechazo en la fase SINTÁCTICO, sin árbol |
| Semántico (E) | Errores SEM-01 a SEM-06, solos y combinados | 10 | Rechazo en la fase SEMÁNTICO con el código y la línea esperados |
| Integración (I) | Programa completo de la demostración «Estrellita» | 1 | Pasan las tres fases sin errores |
| Generación de código (G) | IR generado con LLVM (Hito 3) | Pendiente | El IR es válido y el audio coincide con las notas, el tempo y la onda del programa |

**Orden de ejecución:** ejecutar antes las celdas de ANTLR, de los analizadores y de la demostración «Estrellita».

Función de compilación completa y registro de cobertura

`compilar_soundspec` ejecuta las tres fases en orden (léxico → sintáctico → semántico) y devuelve la fase que rechaza el programa, junto con los códigos y las líneas de los errores. `CoberturaReglas` recorre el árbol de cada programa aceptado para registrar qué reglas de la gramática se usaron.

In [ ]:
from antlr4 import InputStream, ParseTreeWalker
from antlr4.tree.Tree import ParseTreeListener
import pandas as pd
import re

# Verifica que las celdas de los analizadores ya se hayan ejecutado
requisitos_validacion = ("analizar_sintaxis", "Declaraciones", "Verificador",
                         "SoundSpecLexer", "SoundSpecParser")
faltantes_validacion = [n for n in requisitos_validacion if n not in globals()]
if faltantes_validacion:
    raise RuntimeError("Ejecuta primero las celdas anteriores del notebook. Faltan: "
                       + ", ".join(faltantes_validacion))

class CoberturaReglas(ParseTreeListener):
    # Registra cada regla visitada del árbol y la alternativa usada en trackProp y effectProp
    def __init__(self, reglas, alternativas):
        self.reglas, self.alternativas = reglas, alternativas

    def enterEveryRule(self, ctx):
        nombre = SoundSpecParser.ruleNames[ctx.getRuleIndex()]
        self.reglas.add(nombre)
        if nombre in ("trackProp", "effectProp"):
            # El primer token indica la alternativa: bpm, wave, notes, delay o decay
            self.alternativas.add(f"{nombre}: {ctx.start.text}")

def compilar_soundspec(cod_fuente, reglas=None, alternativas=None):
    """Ejecuta léxico -> sintáctico -> semántico y devuelve la fase que rechaza."""
    # Fases léxica y sintáctica: si fallan, no se ejecuta el análisis semántico
    resultado = analizar_sintaxis(cod_fuente)
    if not resultado["valido"]:
        fase = resultado["errores"][0]["fase"] if resultado["errores"] else "SINTÁCTICO"
        return {"fase": fase, "codigos": set(), "lineas": set(),
                "mensajes": [e["mensaje"] for e in resultado["errores"]]}

    # Fase semántica: los dos recorridos del analizador semántico del proyecto
    tabla, errores = {}, []
    ParseTreeWalker().walk(Declaraciones(tabla, errores), resultado["arbol"])
    ParseTreeWalker().walk(Verificador(tabla, errores), resultado["arbol"])

    # Registro de cobertura de la gramática (solo para árboles válidos)
    if reglas is not None:
        ParseTreeWalker().walk(CoberturaReglas(reglas, alternativas), resultado["arbol"])

    # Extrae el código (SEM-0X) y la línea de cada mensaje de error
    if errores:
        return {"fase": "SEMÁNTICO",
                "codigos": {re.search(r"\[(SEM-\d+)\]", e).group(1) for e in errores},
                "lineas": {int(re.search(r"línea (\d+)", e).group(1)) for e in errores},
                "mensajes": errores}
    return {"fase": "ACEPTADO", "codigos": set(), "lineas": set(), "mensajes": []}

print("Función compilar_soundspec lista.")

Función compilar_soundspec lista.


Casos de aceptación (A): programas válidos

Cada caso tiene la forma `(ID, categoría, descripción, código fuente, fase esperada, códigos esperados, línea esperada)`. Estos programas deben pasar las tres fases sin errores; incluyen valores frontera como la nota E0 (20.60 Hz) y un `mix` declarado antes que su pista.

In [ ]:
casos_aceptacion = [
    # --- A01: pista con bpm, wave y notes ---
    ("A01", "Aceptación", "Pista con las tres propiedades",
     "track Bajo { bpm: 120; wave: SAWTOOTH; notes: [C2:1/4, E2:1/4, G2:1/2]; }", "ACEPTADO", set(), None),
    # --- A02: notas sin «:», con sostenidos y bemoles ---
    ("A02", "Aceptación", "Notas sin «:», sostenidos y bemoles",
     "track Alt { notes: [C#4 1/2, Db4 1/4, Bb3 1/4]; }", "ACEPTADO", set(), None),
    # --- A03: efecto con sus dos parámetros ---
    ("A03", "Aceptación", "Efecto con delay y decay",
     "effect Eco { delay: 300ms; decay: 0.85; }", "ACEPTADO", set(), None),
    # --- A04: cadena de dos efectos en un mix ---
    ("A04", "Aceptación", "Mix con cadena de dos efectos",
     "track M { wave: SINE; }\neffect Eco { delay: 80ms; }\neffect Sala { decay: 0.3; }\nmix S = M -> Eco -> Sala;",
     "ACEPTADO", set(), None),
    # --- A05: referencia hacia adelante, resuelta por el segundo recorrido ---
    ("A05", "Aceptación", "Mix declarado antes que su pista (dos recorridos)",
     "mix S = M;\ntrack M { bpm: 90; }", "ACEPTADO", set(), None),
    # --- A06: las duraciones suman 2 compases completos ---
    ("A06", "Aceptación", "Duraciones que completan dos compases",
     "track Larga { notes: [C4 3/2, G4 1/2]; }", "ACEPTADO", set(), None),
    # --- A07: comentarios y propiedades en otro orden ---
    ("A07", "Aceptación", "Comentarios y propiedades en otro orden",
     "// comentario\ntrack T { wave: TRIANGLE; bpm: 100; } // fin", "ACEPTADO", set(), None),
    # --- A08: frontera inferior del rango audible (20 Hz) ---
    ("A08", "Aceptación", "Frontera inferior audible: E0 (20.60 Hz)",
     "track Grave { notes: [E0 1/1]; }", "ACEPTADO", set(), None),
    # --- A09: nota más aguda que admite el lexer ---
    ("A09", "Aceptación", "Onda SQUARE y nota más aguda B8 (7902 Hz)",
     "track Agudo { wave: SQUARE; notes: [B8 1/1]; }", "ACEPTADO", set(), None),
]
print(f"Casos de aceptación definidos: {len(casos_aceptacion)}")

Casos de aceptación definidos: 9


Casos léxicos (L): caracteres no reconocidos

Estas entradas contienen caracteres que no pertenecen a ningún token. Deben rechazarse en la fase LÉXICO, antes de construir el árbol.

In [ ]:
casos_lexicos = [
    # --- L01: símbolo fuera del alfabeto del lenguaje ---
    ("L01", "Léxico", "Carácter no reconocido «@»",
     "track T { bpm: 120; } @", "LÉXICO", set(), None),
    # --- L02: ID solo admite letras ASCII, dígitos y guion bajo ---
    ("L02", "Léxico", "Identificador con tilde",
     "track Atmósfera { bpm: 120; }", "LÉXICO", set(), None),
    # --- L03: símbolo no reconocido dentro de una lista de notas ---
    ("L03", "Léxico", "Símbolo «$» dentro de una lista",
     "track T { notes: [C4 $1/1]; }", "LÉXICO", set(), None),
]
print(f"Casos léxicos definidos: {len(casos_lexicos)}")

Casos léxicos definidos: 3


Casos sintácticos (S): estructura inválida

Los tokens son válidos, pero su orden no cumple la gramática `SoundSpecParser.g4`. Deben rechazarse en la fase SINTÁCTICO y no devolver árbol.

In [ ]:
casos_sintacticos = [
    # --- S01: program exige al menos una declaración ---
    ("S01", "Sintáctico", "Programa vacío", "", "SINTÁCTICO", set(), None),
    # --- S02 a S04: separadores y delimitadores ---
    ("S02", "Sintáctico", "«=» en lugar de «:»", "track T { bpm = 120; }", "SINTÁCTICO", set(), None),
    ("S03", "Sintáctico", "Falta «;»", "track T { bpm: 120 }", "SINTÁCTICO", set(), None),
    ("S04", "Sintáctico", "Falta «}»", "effect E { delay: 300ms;", "SINTÁCTICO", set(), None),
    # --- S05 y S06: listas de notas ---
    ("S05", "Sintáctico", "Falta coma entre notas", "track T { notes: [C4 1/2 D4 1/2]; }", "SINTÁCTICO", set(), None),
    ("S06", "Sintáctico", "Lista de notas vacía", "track T { notes: []; }", "SINTÁCTICO", set(), None),
    # --- S07: un efecto requiere al menos una propiedad ---
    ("S07", "Sintáctico", "Bloque de efecto vacío", "effect E { }", "SINTÁCTICO", set(), None),
    # --- S08: después de «->» debe ir un identificador ---
    ("S08", "Sintáctico", "«->» sin efecto", "mix S = T -> ;", "SINTÁCTICO", set(), None),
    # --- S09 y S10: token de tipo incorrecto ---
    ("S09", "Sintáctico", "bpm con decimal", "track T { bpm: 120.5; }", "SINTÁCTICO", set(), None),
    ("S10", "Sintáctico", "delay sin unidad ms", "effect E { delay: 300; }", "SINTÁCTICO", set(), None),
    # --- S11: EOF exige consumir toda la entrada ---
    ("S11", "Sintáctico", "Texto sobrante después del programa", "track T { bpm: 120; } sobrante", "SINTÁCTICO", set(), None),
]
print(f"Casos sintácticos definidos: {len(casos_sintacticos)}")

Casos sintácticos definidos: 11


Casos semánticos (E): errores SEM-01 a SEM-06

La estructura es válida, pero el programa viola una regla de significado. Además de la fase, se comprueba el código de error y la línea en la que se reporta.

In [ ]:
casos_semanticos = [
    # --- E01: SEM-01, identificador redeclarado ---
    ("E01", "Semántico", "track y effect con el mismo nombre",
     "track M { bpm: 90; }\neffect M { decay: 0.5; }", "SEMÁNTICO", {"SEM-01"}, 2),
    # --- E02 y E03: SEM-02, referencias no declaradas ---
    ("E02", "Semántico", "Pista fuente no declarada",
     "mix S = Guitarra;", "SEMÁNTICO", {"SEM-02"}, 1),
    ("E03", "Semántico", "Efecto no declarado en la cadena",
     "track M { bpm: 90; }\nmix S = M -> Reverb;", "SEMÁNTICO", {"SEM-02"}, 2),
    # --- E04: SEM-03, nota fuera del rango audible ---
    ("E04", "Semántico", "Nota bajo 20 Hz: D#0 (19.45 Hz)",
     "track G { notes: [D#0 1/1]; }", "SEMÁNTICO", {"SEM-03"}, 1),
    # --- E05: SEM-04, compás incompleto ---
    ("E05", "Semántico", "Compás incompleto (3/4)",
     "track M { notes: [C4 1/4, D4 1/4, E4 1/4]; }", "SEMÁNTICO", {"SEM-04"}, 1),
    # --- E06 y E07: SEM-05, categoría incorrecta en mix ---
    ("E06", "Semántico", "Fuente del mix es un effect",
     "effect E { delay: 80ms; }\nmix S = E;", "SEMÁNTICO", {"SEM-05"}, 2),
    ("E07", "Semántico", "Cadena del mix contiene un track",
     "track M { bpm: 90; }\nmix S = M -> M;", "SEMÁNTICO", {"SEM-05"}, 2),
    # --- E08 y E09: SEM-06, duración inválida ---
    ("E08", "Semántico", "Duración con denominador 0",
     "track M { notes: [C4 1/0, D4 1/1]; }", "SEMÁNTICO", {"SEM-06"}, 1),
    ("E09", "Semántico", "Duración con numerador 0",
     "track M { notes: [C4 0/4, D4 1/1]; }", "SEMÁNTICO", {"SEM-06"}, 1),
    # --- E10: varios errores en un mismo programa ---
    ("E10", "Semántico", "Varios errores en un mismo programa",
     "track M { notes: [C0 1/4]; }\neffect M { decay: 0.5; }\nmix S = X -> M;",
     "SEMÁNTICO", {"SEM-01", "SEM-02", "SEM-03", "SEM-04", "SEM-05"}, None),
]
print(f"Casos semánticos definidos: {len(casos_semanticos)}")

Casos semánticos definidos: 10


Caso de integración (I) y suite completa

Se agrega el programa completo de la demostración «Estrellita» (si su celda fue ejecutada) y se unen todas las categorías en una sola lista de casos.

In [ ]:
casos_integracion = []
if "cancion_estrellita" in globals():
    casos_integracion.append(("I01", "Integración", "Programa completo «Estrellita» (42 notas)",
                              cancion_estrellita, "ACEPTADO", set(), None))
else:
    print("Aviso: ejecuta la celda de «Estrellita» para incluir el caso de integración.")

# Suite completa del plan de validación
casos_validacion = (casos_aceptacion + casos_lexicos + casos_sintacticos
                    + casos_semanticos + casos_integracion)
print(f"Total de casos en la suite: {len(casos_validacion)}")

Total de casos en la suite: 34


Ejecución de los casos y reporte de resultados

Cada caso se compila con `compilar_soundspec`. Un caso se aprueba si coincide la fase y, cuando se indican, también los códigos de error y la línea. Durante la ejecución se registran las reglas, alternativas, tokens y errores semánticos cubiertos.

In [ ]:
# Conjuntos para las métricas de cobertura
reglas_visitadas, alternativas_visitadas, tokens_vistos = set(), set(), set()
detectados = set()
filas = []

for cid, categoria, descripcion, codigo, fase_esp, codigos_esp, linea_esp in casos_validacion:
    # Compila el caso y acumula los errores semánticos detectados
    obtenido = compilar_soundspec(codigo, reglas_visitadas, alternativas_visitadas)
    detectados |= obtenido["codigos"]

    # Cobertura léxica
    lexer_cobertura = SoundSpecLexer(InputStream(codigo))
    lexer_cobertura.removeErrorListeners()
    tokens_vistos.update(lexer_cobertura.symbolicNames[t.type]
                         for t in lexer_cobertura.getAllTokens())

    # Compara obtenido con lo esperado (fase, códigos y línea)
    ok = (obtenido["fase"] == fase_esp
          and (not codigos_esp or obtenido["codigos"] == codigos_esp)
          and (linea_esp is None or linea_esp in obtenido["lineas"]))
    filas.append({
        "ID": cid, "Categoría": categoria, "Caso": descripcion,
        "Esperado": fase_esp + (f" {sorted(codigos_esp)}" if codigos_esp else ""),
        "Obtenido": obtenido["fase"] + (f" {sorted(obtenido['codigos'])}" if obtenido["codigos"] else ""),
        "Resultado": "APROBADO" if ok else "FALLIDO",
    })

reporte = pd.DataFrame(filas)
pd.set_option("display.max_colwidth", 60)
display(reporte)

,ID,Categoría,Caso,Esperado,Obtenido,Resultado
0,A01,Aceptación,Pista con las tres propiedades,ACEPTADO,ACEPTADO,APROBADO
1,A02,Aceptación,"Notas sin «:», sostenidos y bemoles",ACEPTADO,ACEPTADO,APROBADO
2,A03,Aceptación,Efecto con delay y decay,ACEPTADO,ACEPTADO,APROBADO
3,A04,Aceptación,Mix con cadena de dos efectos,ACEPTADO,ACEPTADO,APROBADO
4,A05,Aceptación,Mix declarado antes que su pista (dos recorridos),ACEPTADO,ACEPTADO,APROBADO
5,A06,Aceptación,Duraciones que completan dos compases,ACEPTADO,ACEPTADO,APROBADO
6,A07,Aceptación,Comentarios y propiedades en otro orden,ACEPTADO,ACEPTADO,APROBADO
7,A08,Aceptación,Frontera inferior audible: E0 (20.60 Hz),ACEPTADO,ACEPTADO,APROBADO
8,A09,Aceptación,Onda SQUARE y nota más aguda B8 (7902 Hz),ACEPTADO,ACEPTADO,APROBADO
9,L01,Léxico,Carácter no reconocido «@»,LÉXICO,LÉXICO,APROBADO


Criterios de aceptación y métricas de cobertura

- El 100 % de los casos obtiene el resultado esperado.
- Cada error semántico (SEM-01 a SEM-06) es detectado por al menos un caso.
- Los casos aceptados recorren todas las reglas de la gramática y todas las alternativas de `trackProp` y `effectProp`.
- Los casos reconocen todos los tokens del lexer (sin contar `WS` y `COMMENT`, que se descartan).
- Un error léxico o sintáctico detiene el proceso antes del análisis semántico.

In [ ]:
# Resumen de casos aprobados por categoría
print("RESUMEN POR CATEGORÍA")
resumen = (reporte.assign(aprobado=reporte["Resultado"].eq("APROBADO"))
           .groupby("Categoría", sort=False)["aprobado"].agg(["sum", "count"]))
for categoria, (aprobados, total) in resumen.iterrows():
    print(f"  {categoria:<12} {aprobados}/{total} casos aprobados")

# Universos de referencia para la cobertura
codigos_sem = [f"SEM-0{i}" for i in range(1, 7)]
tokens_lexer = {n for n in SoundSpecLexer.symbolicNames[1:] if n not in ("WS", "COMMENT")}
alternativas_totales = {"trackProp: bpm", "trackProp: wave", "trackProp: notes",
                        "effectProp: delay", "effectProp: decay"}

# Métricas de los criterios de aceptación
print("\nCRITERIOS DE ACEPTACIÓN")
metricas = [
    ("Casos aprobados", reporte["Resultado"].eq("APROBADO").sum(), len(reporte)),
    ("Errores semánticos cubiertos (SEM-01 a SEM-06)", len(detectados & set(codigos_sem)), len(codigos_sem)),
    ("Reglas de la gramática ejercitadas", len(reglas_visitadas & set(SoundSpecParser.ruleNames)), len(SoundSpecParser.ruleNames)),
    ("Alternativas de propiedades ejercitadas", len(alternativas_visitadas & alternativas_totales), len(alternativas_totales)),
    ("Tokens del lexer reconocidos", len(tokens_vistos & tokens_lexer), len(tokens_lexer)),
]
for nombre, valor, total in metricas:
    print(f"  {nombre:<48} {valor}/{total} ({100 * valor / total:.0f} %)")

# Verificación final: la celda falla si algún criterio no se cumple
fallidos = reporte[reporte["Resultado"] == "FALLIDO"]
assert fallidos.empty, f"Casos fallidos: {', '.join(fallidos['ID'])}"
assert detectados >= set(codigos_sem), "Hay errores semánticos sin caso de prueba"
print("\nEl compilador cumple el plan de validación del front end.")

RESUMEN POR CATEGORÍA
  Aceptación   9/9 casos aprobados
  Léxico       3/3 casos aprobados
  Sintáctico   11/11 casos aprobados
  Semántico    10/10 casos aprobados
  Integración  1/1 casos aprobados

CRITERIOS DE ACEPTACIÓN
  Casos aprobados                                  34/34 (100 %)
  Errores semánticos cubiertos (SEM-01 a SEM-06)   6/6 (100 %)
  Reglas de la gramática ejercitadas               9/9 (100 %)
  Alternativas de propiedades ejercitadas          5/5 (100 %)
  Tokens del lexer reconocidos                     24/24 (100 %)

El compilador cumple el plan de validación del front end.
